## Implementing a Basic NN in pytorch

- Imports

In [1]:
import torch
import torch.nn as nn    ## Creates NN
import torch.optim as optim     ## Gets Optimizers like SDG and ADAM
import torch.nn.functional as F        ## All the functions that dont have any parameters like relu etc
from torch.utils.data import DataLoader     ## Easy dataset management, batches to train on 
import torchvision.datasets as datasets         ## Standard Datasets
import torchvision.transforms as transforms         ## Transformations to perform on datasets

## FCNN

- The Flow is like we have 64 batch size (64 images in a single flow) 
- All with 784 pixels (28x28) flattened as nn.Linear can only handle 1D operation
- These inputs are (64, 784)
- These 64 images are given to layer 1 which converts these pixels into 50 then apply relu and then layer 2 converts them into 10 classes each for 1 image out of all 64 images 

- The model returns the shape [64, 10]

In [2]:
class FCNN(nn.Module):
    def __init__(self, input_size, classes):
        super(FCNN, self).__init__()        ## super tells pytorch to initialize the base stuff and keep track of parameters
        self.fc1 = nn.Linear(input_size, 50)
        self.fc2 = nn.Linear(50, classes)


    def forward(self, x):
        x = F.relu(self.fc1(x))         ## Data flows from fc1 --> relu --> fc2
        x = self.fc2(x)
        return x

## Set Device
- In this case we have GPU 

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Hyperparameters

In [9]:
input_size = 28*28
classes = 10
learning_rate = 0.001 ## For ADAM we usually take smaller for SGD we can start with 0.01
batch = 64
epochs = 10

## Dataset

In [10]:
train_dataset = datasets.MNIST(root='datasets/', download=True, train=True, transform=transforms.ToTensor())
train_loader = DataLoader(dataset=train_dataset, batch_size=batch, shuffle=True)

test_dataset = datasets.MNIST(root='datasets/', train=False, download=True, transform=transforms.ToTensor())
test_loader = DataLoader(dataset=test_dataset, shuffle=False, batch_size=batch)

## Init Model

In [11]:
model = FCNN(input_size=input_size, classes=classes).to(device)

## Loss and Optimizer

In [12]:
loss_func = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

## Training

In [13]:
for i in range(epochs):
    for batch_idx, (data, targets) in enumerate(train_loader):
        ## Get data to cuda 
        data = data.to(device=device)
        targets = targets.to(device=device)
        
        data = data.flatten(1)


        ## Forward 
        score = model(data)
        loss = loss_func(score, targets)

        ## Back prop
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


## EVAL

In [17]:
def check_acc(loader, model):
    num_correct = 0
    num_samples = 0
    model.eval()


    with torch.no_grad():
        for x, y in loader:
            x = x.to(device=device)
            y = y.to(device=device)
            x = x.flatten(1)


            scores = model(x)
            _, predictions = scores.max(1) ## max of 2nd dimenstion (64x10)
            num_correct += (predictions==y).sum()
            num_samples += predictions.size(0)

        return (f'{num_correct} / {num_samples} with accuracy {float(num_correct)/float(num_samples)*100:.2f}')

    model.train()



print(f'This is for Training : {check_acc(train_loader, model)}')
print(f'This is for Testing data: {check_acc(test_loader, model)}') 


This is for Training : 59184 / 60000 with accuracy 98.64
This is for Testing data: 9720 / 10000 with accuracy 97.20
